In [1]:
import json
from datasets import load_dataset

from evalforge.utils import pprint

## Load the dataset

In [2]:
# https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

DATASET_NAME = "Amazon-Reviews-2023"
CATEGORY = "Clothing_Shoes_and_Jewelry" # beware, this is huge!


In [3]:
category = CATEGORY

# def load_category(category):
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                    f"raw_review_{category}", split="full", trust_remote_code=True)
dataset_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                        f"raw_meta_{category}", split="full", trust_remote_code=True)
print(f"Loaded {len(dataset)} reviews and {len(dataset_meta)} metadata")
pprint(dataset[0])
print("-"*100)
pprint(dataset_meta[0])
# return dataset, dataset_meta

Loaded 66033346 reviews and 7218481 metadata
{
    "rating": 3.0,
    "title": "Arrived Damaged : liquid in hub locker!",
    "text": "Unfortunately Amazon in their wisdom (cough, cough) decided to ship the snowsuit in a vinyl bag with holes in it!  There was no other bag to protect the snowsuit inside vinyl bag with all the holes.  This is what happened:  Arrived in hub locker. It was the very top locker. Opened it & pulled the pkg out getting a very wet & nasty surprise at the same time. My senses were assaulted. Smells like tea tree oil. Feels like conditioner or lotion.  I can\u2019t understand how the delivery person a) didn\u2019t smell that mess when they shoved the pkg in b) didn\u2019t see the mess when they shoved it in - tho if they were short I guess that would explain it bc I\u2019m 5\u201910\u201d & I didn\u2019t see it until the pkg was in my hands. The locker was up high & dark, but I could smell it the minute I walked into the hub locker room. I happen to be extremely 

In [4]:
pprint(dataset_meta[0])

{
    "main_category": "AMAZON FASHION",
    "title": "BALEAF Women's Long Sleeve Zip Beach Coverup UPF 50+ Sun Protection Hooded Cover Up Shirt Dress with Pockets",
    "average_rating": 4.2,
    "rating_number": 422,
    "features": [
        "90% Polyester, 10% Spandex",
        "Zipper closure",
        "Machine Wash",
        "Long sleeve sun protection coverups--UPF 50+ blocks the sun from burning",
        "Zipped v-neckline--fashionable V neck and smooth 1/4 zipper allows to staying place as you like",
        "Two drop-in side pockets--hold your phone or keys well\uff0cno worries of falling out",
        "Hoodie with non-slip drawcord--Enhancing hooded design is convenient to wrap your face and enough space to put your head and hair easily",
        "A flattering coverups company you spend all day on the beach\uff0ctraveling with lovers or busying around house. Recommended For everyday leisure or daily exercise"
    ],
    "description": [],
    "price": "31.99",
    "images":

## Merge on items and reviews

- `parent_asin` is the ASIN of the product
- `title_meta` is the title of the product
- `title_review` is the title of the review

We are going to sample from the metadata dataset as it contains the actual products data. Once we sample here, we can merge on the reviews dataset.

In [5]:
SAMPLE_SIZE = 10_000

def filter_meta(x):
    cond = (x["parent_asin"] is not None and
            x["title"] is not None and 
            x["description"] is not None and
            x["average_rating"] > 3.0 and
            x["rating_number"] > 10)
    return cond

dataset_meta_filtered = dataset_meta.filter(filter_meta, num_proc=16)

dataset_meta_sample = dataset_meta_filtered.shuffle(seed=42).select(range(SAMPLE_SIZE))

Filter (num_proc=16): 100%|██████████| 7218481/7218481 [00:31<00:00, 227885.34 examples/s]


In [12]:
ids = dataset_meta_sample["parent_asin"]

In [19]:
#let's filter dataset on parent_asin in dataset_meta_sample
dataset_filtered = dataset.filter(lambda x: x["parent_asin"] in ids, num_proc=16)
dataset_filtered


Filter (num_proc=16): 100%|██████████| 66033346/66033346 [05:53<00:00, 186782.28 examples/s]


Dataset({
    features: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'],
    num_rows: 151790
})

## Let's merge!

In [20]:
import pandas as pd

# Convert reviews and metadata datasets to pandas DataFrames
metadata_df = pd.DataFrame(dataset_meta_sample)
reviews_df = pd.DataFrame(dataset_filtered)

print(f"Before filtering: {len(metadata_df)} metadata, {len(reviews_df)} reviews")

Before filtering: 10000 metadata, 151790 reviews


In [21]:
# Let's join them on `parent_asin`
merged_df = pd.merge(metadata_df, reviews_df, on="parent_asin", how="inner", suffixes=("_meta", "_review"))
merged_df.head()

,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
0,AMAZON FASHION,Arthur's Jewelry Sterling Silver 925 Hawaiian ...,4.6,12,"[Flower size: 12mm, Weight: approx. 3.6 grams,...",[Sterling silver 925 Hawaiian 12mm plumeria fl...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Arthur's Jewelry,...,None,5.0,Loved it!!!,Stunning ring,[],B01MRO2ZIM,AEFQ3PF3GFQ3KAOXIZ6JXRRNUPOQ,1594181533842,0,True
1,AMAZON FASHION,Lesubuy Shiny Mermaid Tail Fish Scales Women's...,4.0,190,"[Pull On closure, Hand Wash Only, ✅Please Chec...",[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Lesubuy,...,None,5.0,Size up 1,I normally wear a large/xl I ordered 2x and it...,[],B01NAL7FYO,AFKHUZ4SSKXIKN3OMLSEL2CCAHQQ,1547483147595,13,True
2,AMAZON FASHION,Lesubuy Shiny Mermaid Tail Fish Scales Women's...,4.0,190,"[Pull On closure, Hand Wash Only, ✅Please Chec...",[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Lesubuy,...,None,5.0,looks great,loved it and was perfect for my event and the ...,[],B01NAL7FYO,AH2GPGINMITJDS6B626JPRIYKSYA,1566827903469,0,True
3,AMAZON FASHION,ThunderFit Mens Silicone Wedding Rings Wedding...,4.5,4048,"[A MUST FOR AN ACTIVE LIFESTYLE – Workout, lif...",[],11.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",ThunderFit,...,None,5.0,Five Stars,Fit perfectly and super comfortable. Colors we...,"[{'attachment_type': 'IMAGE', 'large_image_url...",B06WVGBC9X,AFYPJRJ65B4744WW2OBZVTDCXZBA,1534030211156,0,True
4,AMAZON FASHION,ThunderFit Mens Silicone Wedding Rings Wedding...,4.5,4048,"[A MUST FOR AN ACTIVE LIFESTYLE – Workout, lif...",[],11.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",ThunderFit,...,None,5.0,Great product and super impressive customer se...,Great product and amazing customer service. Th...,[],B07LBQYB96,AERR6VVLNUWUQYVHFOPWBC5RSZFQ,1573593993910,0,False


In [22]:
len(merged_df)

151790

In [67]:
final_df = merged_df.set_index("parent_asin").sort_index()
final_df.head()

,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
parent_asin,,,,,,,,,,,,,,,,,,,,,
1942618891,Books,Defending Roxanne (Lynyrd Station Protectors -...,4.6,507,[GHOST: Government Hidden Ops Specialty Team. ...,[],17.99,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}","PJ Fiala (Author), Marijane Diodoti (Editor)",...,{'avatar': 'https://m.media-amazon.com/images/...,5.0,A must read! Page turner!!!,This had to be one of my favorites so far!!! I...,[],1942618891,AGVE6TKN4IPMBTQDPYCZCUPRLODA,1593430871218,0,True
1942618891,Books,Defending Roxanne (Lynyrd Station Protectors -...,4.6,507,[GHOST: Government Hidden Ops Specialty Team. ...,[],17.99,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}","PJ Fiala (Author), Marijane Diodoti (Editor)",...,{'avatar': 'https://m.media-amazon.com/images/...,5.0,What's not to love??,Absolutely loved this book. This book has it a...,[],1942618891,AE3XSPLBKJJLSVB2TZDGIZHPDWXQ,1571417009747,1,False
1942618891,Books,Defending Roxanne (Lynyrd Station Protectors -...,4.6,507,[GHOST: Government Hidden Ops Specialty Team. ...,[],17.99,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}","PJ Fiala (Author), Marijane Diodoti (Editor)",...,{'avatar': 'https://m.media-amazon.com/images/...,5.0,Awesome read!,Love this series and The men of GHOST very muc...,[],1942618891,AE3N5EN2S3V4G6RE3KOALOCYOSJA,1571671784069,1,False
B0000DZA27,AMAZON FASHION,Saucony Women's ProGrid Stabil CS Running Shoe,4.2,35,"[100% Mesh and synthetic, Rubber sole, Heel Ca...",[Saucony is among the most respected names in ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Saucony,...,None,5.0,Great Shoe for Overpronator,I severely overpronate. These are great motion...,[],B0000DZA27,AH45CHODQHC6G3TU2RRPJB6QUG3Q,1288784653000,1,False
B0000DZA27,AMAZON FASHION,Saucony Women's ProGrid Stabil CS Running Shoe,4.2,35,"[100% Mesh and synthetic, Rubber sole, Heel Ca...",[Saucony is among the most respected names in ...,None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Saucony,...,None,5.0,Fits right!,"This was actually for my sister, suffering fro...",[],B0000DZA27,AECXEYNNYWNSLR67QBOZPSZZ7LBA,1314946880000,0,True


In [68]:
# I want to index on the parent_asin but with an integer index

first_item = final_df.index.get_level_values("parent_asin").unique()[3]
print(first_item)
final_df.loc[first_item]

B00021CAX2


,main_category,title_meta,average_rating,rating_number,features,description,price,images_meta,videos,store,...,author,rating,title_review,text,images_review,asin,user_id,timestamp,helpful_vote,verified_purchase
parent_asin,,,,,,,,,,,,,,,,,,,,,
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,Great watch,I bought this as a gift and it is loved. It's ...,[],B00021CAX2,AFQM7WXGWJTE3Z2GTVYUUY4ZGHWQ,1639147732268,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,A good choice in wrist watches.,I recently lost a similar Timex Expedition wri...,[],B00021CAX2,AH2GM5DDEKJLYPXLW3NAWNY2YXTQ,1668628046025,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,Takes a licking and keeps on ticking!,I have had the previous model of this exact sa...,[],B00021CAX2,AG2ZV2PXSYQPSUOORYKRGERSHBEQ,1220785172000,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,4.0,I love this watch!,I do love this watch. It has 3 alarms which I ...,[],B00021CAX2,AGPXY4ZN3QZ5EEMZQCINSALV4TNQ,1679142306680,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,Five Stars,My son loves his watch. He likes the style.,[],B00021CAX2,AF3QWU4MYW43KUOD5SHU4R7VIZDQ,1410735079000,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,4.0,replacement for the last one.,This is my 3rd or 4th of these . I get a new o...,[],B00021CAX2,AGVSKFLVHGYHW2G7D36JEVWDC6GA,1404656889000,0,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,A great watch!,This is simply a great watch. I really apprec...,[],B00021CAX2,AEZAU3YWUCJVL4LZWE4CIXFT5YUQ,1183890834000,3,False
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resistant leather and black moist...,48.3,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Timex Men's Expedition Atlantis', ...",Timex,...,None,5.0,Men's Timex Digital Expedition Chrono Alarm Ti...,This is truly a great watch. I appreciate the ...,[],B00021CAX2,AHTASLDEHLYSACAD6AGJA4XGUWGQ,1401965013000,1,True
B00021CAX2,AMAZON FASHION,Men's Timex Digital Expedition Chrono Alarm Ti...,4.3,62,"[100-hour chronograph, 100-hour countdown time...",[Brown water-resista

## Save to disk!

We will flatten the dataframe and save it to disk.

In [72]:
final_df.columns

Index(['main_category', 'title_meta', 'average_rating', 'rating_number',
       'features', 'description', 'price', 'images_meta', 'videos', 'store',
       'categories', 'details', 'bought_together', 'subtitle', 'author',
       'rating', 'title_review', 'text', 'images_review', 'asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase'],
      dtype='object')

In [78]:
import numpy as np

# Define which columns we want
METADATA_COLUMNS = [
    "main_category",
    "title_meta",
    "description",
    "average_rating",
    "rating_number",
    "asin",
    "features",
    "price",
    "images_meta",
    "videos",
    "store",
    "categories",
    "details",
    "bought_together",
    "subtitle",
    "author"
]

REVIEW_COLUMNS = [
    "rating",
    "title_review",
    "text",
    "images_review",
    "user_id",
    "timestamp",
    "helpful_vote",
    "verified_purchase"
]

def create_product_entry(group, metadata_columns=METADATA_COLUMNS, review_columns=REVIEW_COLUMNS):
    # Get metadata from first row
    metadata = {
        "parent_asin": group.index[0],  # Get parent_asin from index
        **{col.replace('_meta', ''): group[col].iloc[0]  # Remove _meta suffix in output
           for col in metadata_columns}
    }
    
    # Create list of reviews
    reviews = group.apply(
        lambda x: {col.replace('_review', ''): x[col]  # Remove _review suffix in output
                  for col in review_columns},
        axis=1
    ).tolist()
    
    return {**metadata, "reviews": reviews}

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.int_, np.intc, np.intp, np.int8,
                          np.int16, np.int32, np.int64, np.uint8,
                          np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# Process each group and save to JSONL
with open("product_data.jsonl", "w") as f:
    for parent_asin, group in final_df.groupby(level=0):
        product_entry = create_product_entry(group)
        f.write(json.dumps(product_entry, cls=NumpyEncoder) + "\n")